# 02 — DataFrames: catálogo de precios PROFECO (`all_data.csv`)

**Objetivo:** pasar de RDDs a la DataFrame API, leer datos reales, y aprender a leer
el plan de ejecución con `.explain()` — la herramienta de diagnóstico más usada en el
resto del curso.

**Dataset:** `all_data.csv` (~19.7 GB) — catálogo completo de "Quién es Quién en los
Precios" de PROFECO. Columnas: `producto`, `presentacion`, `marca`, `categoria`,
`catalogo`, `precio`, `fechaRegistro`, `cadenaComercial`, `giro`, `nombreComercial`,
`direccion`, `estado`, `municipio`, `latitud`, `longitud`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("02_dataframes").getOrCreate()

RUTA = "gs://<TU-BUCKET>/raw/profeco/all_data.csv"

## 1. Lectura

In [ ]:
df = spark.read.csv(RUTA, header=True, inferSchema=True)
df.printSchema()
df.count()

## 2. Transformaciones con la DataFrame API

Pregunta de negocio: ¿cuál es el precio promedio y cuántos productos tiene cada cadena
comercial, por estado? — el tipo de agregación geoespacial/retail que justifica este
dataset.

In [ ]:
resumen = (
    df.filter(F.col("precio") > 0)
    .groupBy("estado", "cadenaComercial")
    .agg(
        F.count("*").alias("num_productos"),
        F.avg("precio").alias("precio_promedio"),
        F.max("precio").alias("precio_max"),
    )
    .orderBy(F.desc("num_productos"))
)

## 3. Leer el plan físico ANTES de ejecutar

Clave para optimizar en sesiones futuras (Sesión 3 de Maestría: Spark Core avanzado).

In [ ]:
print("=== Plan físico (Catalyst) ===")
resumen.explain(mode="formatted")

## 4. Acción: aquí sí se ejecuta todo el plan de arriba

In [ ]:
resumen.show(20, truncate=False)

## 5. Guardar en formato columnar particionado

Preparación directa para la Sesión 6 (Data Lakes / arquitectura medallion): particionar
por `estado` es la columna correcta aquí porque es por la que más se va a filtrar
después (partition pruning).

In [ ]:
resumen.write.mode("overwrite").partitionBy("estado").parquet(
    "gs://<TU-BUCKET>/processed/profeco_resumen/"
)

In [ ]:
spark.stop()